# CS661 Assignment 2: Interactive Isosurface and Histogram Visualization

In [4]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import vtk
from vtk.util import numpy_support
import os

In [5]:

def read_vti_file(filepath):

    #Read VTI file and extract 3D scalar data with coordinate grids.
    #Parameters: filepath : dataType=(str)

    reader = vtk.vtkXMLImageDataReader()  # VTI file reader
    reader.SetFileName(filepath)
    reader.Update()
    
    # Get the ImageData object
    imageData = reader.GetOutput()
    
    # Get dimensions, spacing, and origin
    dims = imageData.GetDimensions()  
    spacing = imageData.GetSpacing() 
    origin = imageData.GetOrigin()   
    
    # Get scalar data
    scalars = imageData.GetPointData().GetScalars()
    scalerArray = numpy_support.vtk_to_numpy(scalars)
    
    # Covert to 3D grid (Fortran order)
    scaler3D = scalerArray.reshape(dims, order='F')
    scalerFlat = scalerArray.copy()
    
    # Get coordinate grids
    x = np.arange(dims[0]) * spacing[0] + origin[0]
    y = np.arange(dims[1]) * spacing[1] + origin[1]
    z = np.arange(dims[2]) * spacing[2] + origin[2]
    
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    
    # Get  data range
    data_min = np.min(scalerFlat)
    data_max = np.max(scalerFlat)
    
    return X, Y, Z, scaler3D, scalerFlat, data_min, data_max

# Load the dataset
vti_file = "mixture.vti"
X, Y, Z, scaler3D, scalerFlat, data_min, data_max = read_vti_file(vti_file)

In [6]:
def create_isosurface(X, Y, Z, scalerData, isovalue, colorscale='Plasma'):
    # Define color range window mapped to entire dataset range
    # to ensure the colormap stretches and shifts as the isovalue changes.
    cmin = np.min(scalerData)
    cmax = np.max(scalerData)
    
    fig = go.Figure(data=go.Isosurface(
        x=X.flatten(), 
        y=Y.flatten(), 
        z=Z.flatten(), 
        
        value=scalerData.flatten(),
        isomin=isovalue,
        isomax=isovalue,
        
        colorscale=colorscale,
        cmin=cmin,
        cmax=cmax,
        
        showscale=False,
        surface_count=1,
        
        caps=dict(x_show=False, y_show=False, z_show=False)
    ))
    
    fig.update_layout(
        width=500,
        height=600,
        margin=dict(l=20, r=20, b=20, t=20),
        scene=dict(
            xaxis_title='x',
            yaxis_title='y',
            zaxis_title='z'
        )
    )
    return fig

def create_histogram(data, title="Distribution", show_range=None):
    
    # Creates a histogram
    """
    Parameters:data: 1D array of values to histogram
            title: Title for the histogram
            show_range: Optional tuple (min, max) to show in subtitle
    
    Returns:fig: Plotly figure object
    """
    # Create histogram
    fig = go.Figure()
    
    fig.add_trace(go.Histogram(
        x=data,
        nbinsx=30,
        name='Frequency',
        marker=dict(
            color='rgba(65, 105, 225, 0.7)',
            line=dict(color='rgba(65, 105, 225, 1.0)', width=1)
        ),
        showlegend=False
    ))
    
    # Generate title with range info
    if show_range:
        title_text = f"{title}<br><sub>Data range: [{show_range[0]:.4f}, {show_range[1]:.4f}]</sub>"
    else:
        title_text = title
    
    fig.update_layout(
        title=title_text,
        xaxis_title="Vortex scalar values",
        yaxis_title="Frequency",
        width=500,
        height=600,
        margin=dict(l=60, r=20, b=60, t=80),
        showlegend=False,
        plot_bgcolor='rgba(240, 240, 255, 0.5)'
    )
    
    # Auto-scale axes
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(200, 200, 200, 0.3)')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(200, 200, 200, 0.3)')
    
    return fig

def get_filtered_data(scalerFlat, isovalue, tolerance=0.25):
    
    #Filter scalar data to values within tolerance of isovalue.
    
    lower_bound = isovalue - tolerance
    upper_bound = isovalue + tolerance
    
    filtered = scalerFlat[(scalerFlat >= lower_bound) & (scalerFlat <= upper_bound)]
    
    if len(filtered) > 0:
        return filtered, np.min(filtered), np.max(filtered)
    else:
        return np.array([isovalue]), isovalue, isovalue

In [7]:
# Initialize state variables
state = {
    'current_isovalue': 0.0,
    'current_filtered_data': scalerFlat.copy(),
    'isosurface_output': None,
    'histogram_output': None
}

# Create output widgets for plots
isosurface_output = widgets.Output()
histogram_output = widgets.Output()


# Isovalue slider
# Map slider range to data min/max
isovalue_slider = widgets.FloatSlider(
    value=0.0,
    min=data_min,
    max=data_max,
    step=(data_max - data_min) / 100,
    description='Isoval:',
    continuous_update=False,  
    orientation='horizontal',
    readout=True,
    readout_format='.2f'
)

#Reset button
reset_button = widgets.Button(
    description='Reset',
    button_style='info',  # 'success', 'info', 'warning', 'danger', 'default'
    tooltip='Reset to initial state (isovalue=0.0, full data histogram)',
    icon='redo'
)

#Label for slider
slider_label = widgets.Label(value='Isovalue:')

In [8]:
def update_visualizations(isovalue, show_full_histogram=False):
    """
    Update both isosurface and histogram based on new isovalue.
    """
    
    # Update state
    state['current_isovalue'] = isovalue
    

    # Create and display isosurface
    with isosurface_output:
        isosurface_output.clear_output(wait=True)
        iso_fig = create_isosurface(X, Y, Z, scaler3D, isovalue, colorscale='Plasma')
        display(iso_fig)
    
    # Create and display histogram
    with histogram_output:
        histogram_output.clear_output(wait=True)
        if show_full_histogram:
            hist_title = "Histogram (Entire Volume Dataset)"
            hist_fig = create_histogram(
                scalerFlat,
                title=hist_title,
                show_range=(data_min, data_max)
            )
            state['current_filtered_data'] = scalerFlat.copy()
        else:
            # Get filtered data within ±0.25 of isovalue
            filtered_data, data_min_filtered, data_max_filtered = get_filtered_data(
                scalerFlat, isovalue, tolerance=0.25
            )
            state['current_filtered_data'] = filtered_data
            hist_title = f"Histogram (Isovalue ±0.25)"
            hist_fig = create_histogram(
                filtered_data,
                title=hist_title,
                show_range=(data_min_filtered, data_max_filtered)
            )
        display(hist_fig)
    
def on_isovalue_change(change):
    #Callback triggered when slider value changes.
    
    new_isovalue = change['new']
    update_visualizations(new_isovalue, show_full_histogram=False)

def on_reset_clicked(b):
    #Callback triggered when Reset button is clicked.
    
    print("Resetting...")
    
    # Temporarily unobserve slider to avoid triggering on_isovalue_change callback
    isovalue_slider.unobserve(on_isovalue_change, names='value')
    isovalue_slider.value = 0.0
    isovalue_slider.observe(on_isovalue_change, names='value')
    
    # Force visualizations update with full dataset histogram
    update_visualizations(0.0, show_full_histogram=True)

# Attach callbacks to widgets
isovalue_slider.observe(on_isovalue_change, names='value')
reset_button.on_click(on_reset_clicked)

In [9]:
# Control panel (horizontal layout with slider and button)
control_panel = widgets.HBox(
    [isovalue_slider, reset_button],
    layout=widgets.Layout(
        width='100%',
        border='1px solid lightgray',
        padding='15px',
        margin_bottom='20px'
    )
)

# Create main content area (side-by-side plots)
plots_area = widgets.HBox(
    [isosurface_output, histogram_output],
    layout=widgets.Layout(
        width='100%',
        border='1px solid lightgray',
        padding='10px'
    )
)

# Create main interface (vertical layout: controls + plots)
main_interface = widgets.VBox(
    [control_panel, plots_area],
    layout=widgets.Layout(
        width='100%',
        border='2px solid steelblue',
        padding='10px'
    )
)

print("Interface layout created. Initializing with isovalue = 0.0...")
print(f"Data statistics:")
print(f"  Full dataset: {len(scalerFlat)} points")
print(f"  Data range: [{data_min:.4f}, {data_max:.4f}]")

Interface layout created. Initializing with isovalue = 0.0...
Data statistics:
  Full dataset: 421875 points
  Data range: [-0.9936, 0.4328]


In [10]:
# Display the main interface

clear_output()
display(main_interface)

# Initialize with default isovalue = 0.0 (and show full dataset histogram)
print("INITIALIZING INTERACTIVE VISUALIZATION")
with isosurface_output:
    update_visualizations(isovalue=0.0, show_full_histogram=True)


INITIALIZING INTERACTIVE VISUALIZATION
